In [19]:
import pandas as pd

signals = pd.read_csv(
    "D:/operational-signal-intelligence-environment/outputs/signal_priority_table.csv"
)

signals["timestamp"] = pd.to_datetime(
    signals["timestamp"]
)

signals.head()

,timestamp,signal_name,severity,reason,supporting_metric,confidence,impact_area,owner,resolution_target,operational_consequence
0,2025-01-02 15:30:00,Demand Spike,High,Demand exceeded spike threshold,38264.0,High,Grid Demand,Regional Controller,2 Hours,Potential grid stress due to elevated demand
1,2025-01-02 16:00:00,Demand Spike,High,Demand exceeded spike threshold,39309.0,High,Grid Demand,Regional Controller,2 Hours,Potential grid stress due to elevated demand
2,2025-01-02 16:30:00,Demand Spike,High,Demand exceeded spike threshold,40412.0,High,Grid Demand,Regional Controller,2 Hours,Potential grid stress due to elevated demand
3,2025-01-02 17:00:00,Demand Spike,High,Demand exceeded spike threshold,41103.0,High,Grid Demand,Regional Controller,2 Hours,Potential grid stress due to elevated demand
4,2025-01-02 17:30:00,Demand Spike,High,Demand exceeded spike threshold,41589.0,High,Grid Demand,Regional Controller,2 Hours,Potential grid stress due to elevated demand


In [20]:
incident_rules = {

    ("Demand Spike", "Supply Stress"):
    "GRID_STRESS_INCIDENT",

    ("Forecast Failure", "Demand Spike"):
    "FORECAST_RISK_INCIDENT",

    ("Low Renewable Window", "High Carbon Period"):
    "SUSTAINABILITY_INCIDENT",

    ("Demand Spike", "High Carbon Period"):
    "PEAK_LOAD_EMISSIONS_INCIDENT",

    ("Supply Stress", "Forecast Failure"):
    "CAPACITY_PLANNING_INCIDENT",

    ("Demand Drop", "Generation Surplus"):
    "OVER_GENERATION_INCIDENT",

    ("Renewable Surge", "Generation Surplus"):
    "CURTAILMENT_RISK_INCIDENT",

    ("Demand Drop", "Low Renewable Window"):
    "DEMAND_ANOMALY_INCIDENT",

    ("Renewable Surge", "Demand Spike"):
    "RENEWABLE_SUPPORT_INCIDENT",

    ("Forecast Failure", "High Carbon Period"):
    "OPERATIONAL_RISK_INCIDENT"
}

In [21]:
grouped = signals.groupby("timestamp")["signal_name"].apply(list)

grouped.head()

timestamp
2025-01-01 00:00:00    [Generation Surplus, Renewable Surge]
2025-01-01 00:30:00    [Generation Surplus, Renewable Surge]
2025-01-01 01:00:00    [Generation Surplus, Renewable Surge]
2025-01-01 01:30:00    [Generation Surplus, Renewable Surge]
2025-01-01 02:00:00    [Generation Surplus, Renewable Surge]
Name: signal_name, dtype: object

In [22]:
incidents = []

incident_counter = 1

for ts, signal_list in grouped.items():

    signal_set = set(signal_list)

    for rule_signals, incident_name in incident_rules.items():

        if set(rule_signals).issubset(signal_set):

            incidents.append({

                "incident_id":
                f"INC-{incident_counter:05d}",

                "timestamp":
                ts,

                "incident_type":
                incident_name,

                "contributing_signals":
                ", ".join(rule_signals)

            })

            incident_counter += 1

In [23]:
incident_df = pd.DataFrame(
    incidents
)

incident_df.head()

,incident_id,timestamp,incident_type,contributing_signals
0,INC-00001,2025-01-01 00:00:00,CURTAILMENT_RISK_INCIDENT,"Renewable Surge, Generation Surplus"
1,INC-00002,2025-01-01 00:30:00,CURTAILMENT_RISK_INCIDENT,"Renewable Surge, Generation Surplus"
2,INC-00003,2025-01-01 01:00:00,CURTAILMENT_RISK_INCIDENT,"Renewable Surge, Generation Surplus"
3,INC-00004,2025-01-01 01:30:00,CURTAILMENT_RISK_INCIDENT,"Renewable Surge, Generation Surplus"
4,INC-00005,2025-01-01 02:00:00,CURTAILMENT_RISK_INCIDENT,"Renewable Surge, Generation Surplus"


In [24]:
severity_map = {

    "GRID_STRESS_INCIDENT": "Critical",

    "FORECAST_RISK_INCIDENT": "High",

    "SUSTAINABILITY_INCIDENT": "Medium",

    "PEAK_LOAD_EMISSIONS_INCIDENT": "High",

    "CAPACITY_PLANNING_INCIDENT": "High",

    "OVER_GENERATION_INCIDENT": "Low",

    "CURTAILMENT_RISK_INCIDENT": "Medium",

    "DEMAND_ANOMALY_INCIDENT": "Medium",

    "RENEWABLE_SUPPORT_INCIDENT": "Low",

    "OPERATIONAL_RISK_INCIDENT": "High"

}

In [25]:
incident_df["severity"] = (
    incident_df["incident_type"]
    .map(severity_map)
)

In [26]:
impact_map = {

    "GRID_STRESS_INCIDENT":
    "Risk of supply shortfall",

    "FORECAST_RISK_INCIDENT":
    "Reduced forecasting reliability",

    "SUSTAINABILITY_INCIDENT":
    "Elevated carbon emissions",

    "PEAK_LOAD_EMISSIONS_INCIDENT":
    "High demand with elevated emissions",

    "CAPACITY_PLANNING_INCIDENT":
    "Generation planning challenges",

    "OVER_GENERATION_INCIDENT":
    "Potential generation inefficiency",

    "CURTAILMENT_RISK_INCIDENT":
    "Potential renewable curtailment",

    "DEMAND_ANOMALY_INCIDENT":
    "Unexpected demand behavior",

    "RENEWABLE_SUPPORT_INCIDENT":
    "Renewables supporting demand",

    "OPERATIONAL_RISK_INCIDENT":
    "Multiple operational concerns"

}

In [27]:
incident_df["business_impact"] = (
    incident_df["incident_type"]
    .map(impact_map)
)

In [28]:
response_map = {

    "GRID_STRESS_INCIDENT":
    "Increase generation capacity and monitor demand",

    "FORECAST_RISK_INCIDENT":
    "Review forecasting models and assumptions",

    "SUSTAINABILITY_INCIDENT":
    "Investigate renewable availability",

    "PEAK_LOAD_EMISSIONS_INCIDENT":
    "Optimize generation mix",

    "CAPACITY_PLANNING_INCIDENT":
    "Review reserve capacity plans",

    "OVER_GENERATION_INCIDENT":
    "Reduce excess generation",

    "CURTAILMENT_RISK_INCIDENT":
    "Evaluate renewable curtailment actions",

    "DEMAND_ANOMALY_INCIDENT":
    "Investigate unusual demand behavior",

    "RENEWABLE_SUPPORT_INCIDENT":
    "Maintain renewable output",

    "OPERATIONAL_RISK_INCIDENT":
    "Perform operational risk assessment"
}

incident_df["recommended_response"] = (
    incident_df["incident_type"]
    .map(response_map)
)

In [29]:
path_map = {

    "Critical":
    "Operator → Supervisor → Regional Controller → Executive",

    "High":
    "Operator → Supervisor → Regional Controller",

    "Medium":
    "Operator → Supervisor",

    "Low":
    "Operator"
}

incident_df["escalation_path"] = (
    incident_df["severity"]
    .map(path_map)
)

In [30]:
incident_df.head()

,incident_id,timestamp,incident_type,contributing_signals,severity,business_impact,recommended_response,escalation_path
0,INC-00001,2025-01-01 00:00:00,CURTAILMENT_RISK_INCIDENT,"Renewable Surge, Generation Surplus",Medium,Potential renewable curtailment,Evaluate renewable curtailment actions,Operator → Supervisor
1,INC-00002,2025-01-01 00:30:00,CURTAILMENT_RISK_INCIDENT,"Renewable Surge, Generation Surplus",Medium,Potential renewable curtailment,Evaluate renewable curtailment actions,Operator → Supervisor
2,INC-00003,2025-01-01 01:00:00,CURTAILMENT_RISK_INCIDENT,"Renewable Surge, Generation Surplus",Medium,Potential renewable curtailment,Evaluate renewable curtailment actions,Operator → Supervisor
3,INC-00004,2025-01-01 01:30:00,CURTAILMENT_RISK_INCIDENT,"Renewable Surge, Generation Surplus",Medium,Potential renewable curtailment,Evaluate renewable curtailment actions,Operator → Supervisor
4,INC-00005,2025-01-01 02:00:00,CURTAILMENT_RISK_INCIDENT,"Renewable Surge, Generation Surplus",Medium,Potential renewable curtailment,Evaluate renewable curtailment actions,Operator → Supervisor


In [31]:
incident_df.shape

(5526, 8)

In [32]:
incident_df["incident_type"].value_counts()

incident_type
CURTAILMENT_RISK_INCIDENT       1606
OVER_GENERATION_INCIDENT        1573
SUSTAINABILITY_INCIDENT         1567
PEAK_LOAD_EMISSIONS_INCIDENT     448
DEMAND_ANOMALY_INCIDENT           88
OPERATIONAL_RISK_INCIDENT         84
GRID_STRESS_INCIDENT              77
FORECAST_RISK_INCIDENT            44
RENEWABLE_SUPPORT_INCIDENT        37
CAPACITY_PLANNING_INCIDENT         2
Name: count, dtype: int64

In [33]:
incident_df.to_csv(
    "D:/operational-signal-intelligence-environment/outputs/incident_table.csv",
    index=False
)

print(
    "Incident table generated successfully"
)

Incident table generated successfully
